# 月次予測ノートブック（日次データ入力 → Xか月先 月末予測）

日次CSVデータを月末値に集約し、**Xか月先の月末MP値**を8つの機械学習モデルで比較予測します。

## 対応モデル（8種類）
1. **Linear Regression**
2. **Ridge Regression**（L2正則化）
3. **Lasso Regression**（L1正則化）
4. **Random Forest**
5. **Gradient Boosting**
6. **XGBoost**
7. **LightGBM**
8. **Prophet**（時系列専用モデル）

## 入力データ形式
| 列 | 説明 |
|---|---|
| `行ラベル` | 日付（例: `2020/4/1`）|
| `合計 / MP` | 予測対象（B列）|
| その他の列 | 自動的に特徴量として使用 |

## 処理フロー
```
日次CSV → 月末値に集約 → ラグ特徴量生成 → 8モデル比較 → Xか月先予測
```

## 1. ライブラリのインストールとインポート

In [ ]:
!pip install xgboost lightgbm prophet -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    mean_absolute_percentage_error, r2_score
)

import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.unicode_minus'] = False

print('ライブラリ読み込み完了')

## 2. 設定

**ここを編集してください** — 列名・予測設定を入力データに合わせて変更してください

In [ ]:
# =============================================================================
# 設定 — 入力データに合わせて編集してください
# =============================================================================

# 列名
DATE_COLUMN   = '行ラベル'     # 日付列名
TARGET_COLUMN = '合計 / MP'    # 予測対象列名（B列 = MP）

# 予測設定
FORECAST_HORIZON = 3           # 何か月先を予測するか（例: 3 = 3か月先の月末MP）
TEST_MONTHS      = 6           # テスト期間（最後の何か月をテストに使うか）

# 表示設定
TARGET_DISPLAY_NAME = 'MP'     # グラフ表示名
TARGET_SCALE        = 1_000_000  # 表示時のスケール（百万円単位: 1_000_000）
TARGET_UNIT         = '百万円'    # 表示単位

# 特徴量から除外する列（日付・ターゲット以外で不要な列があれば追加）
EXCLUDE_COLUMNS = []

# =============================================================================

print('設定完了')
print(f'  日付列     : {DATE_COLUMN}')
print(f'  予測対象列  : {TARGET_COLUMN}')
print(f'  予測先月数  : {FORECAST_HORIZON}か月先')
print(f'  テスト期間  : {TEST_MONTHS}か月')

## 3. データ読み込み

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
if len(uploaded) == 0:
    raise ValueError('エラー: ファイルがアップロードされていません。上のセルを実行してCSVをアップロードしてください。')

filename = list(uploaded.keys())[0]

# エンコーディングを自動判定
df_raw = None
for enc in ['utf-8', 'utf-8-sig', 'cp932', 'shift_jis']:
    try:
        df_raw = pd.read_csv(filename, encoding=enc)
        print(f'エンコーディング: {enc}')
        break
    except UnicodeDecodeError:
        continue

if df_raw is None:
    raise ValueError('CSVの読み込みに失敗しました。ファイルのエンコーディングを確認してください。')

df_raw.columns = df_raw.columns.str.strip()

print(f'\nデータ形状: {df_raw.shape}')
print(f'\n列名一覧 ({len(df_raw.columns)}列):')
for i, col in enumerate(df_raw.columns):
    print(f'  [{i}] {col}')
print(f'\n先頭3行:')
df_raw.head(3)

## 4. 前処理：日次データ → 月次（月末値）集約

In [ ]:
# 必須列の存在確認
if DATE_COLUMN not in df_raw.columns:
    raise ValueError(f'日付列 "{DATE_COLUMN}" が見つかりません。\n利用可能な列: {df_raw.columns.tolist()}')
if TARGET_COLUMN not in df_raw.columns:
    raise ValueError(f'予測対象列 "{TARGET_COLUMN}" が見つかりません。\n利用可能な列: {df_raw.columns.tolist()}')

df = df_raw.copy()

# 日付列をdatetimeに変換
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors='coerce')
invalid_dates = df[DATE_COLUMN].isna().sum()
if invalid_dates > 0:
    print(f'警告: {invalid_dates}行の日付が無効です。これらの行を除外します。')
df = df.dropna(subset=[DATE_COLUMN]).sort_values(DATE_COLUMN).reset_index(drop=True)

# 数値列の変換（カンマ・クォート除去）
numeric_cols = [c for c in df.columns if c != DATE_COLUMN]
for col in numeric_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '').str.replace('"', '').str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── 月末値に集約（各月の最終日の値を使用）──
df['_ym'] = df[DATE_COLUMN].dt.to_period('M')
df_monthly = (
    df.sort_values(DATE_COLUMN)
      .groupby('_ym', sort=True)
      .last()
      .reset_index()
)
# 月初に統一（YYYY-MM-01）
df_monthly[DATE_COLUMN] = df_monthly['_ym'].dt.to_timestamp()
df_monthly = df_monthly.drop(columns=['_ym']).sort_values(DATE_COLUMN).reset_index(drop=True)

print(f'日次データ: {len(df):,}行')
print(f'月次データ（月末値）: {len(df_monthly)}か月')
print(f'期間: {df_monthly[DATE_COLUMN].min().strftime("%Y-%m")} ～ {df_monthly[DATE_COLUMN].max().strftime("%Y-%m")}')
print(f'\n月次データ（先頭5行）:')
df_monthly[[DATE_COLUMN, TARGET_COLUMN]].head()

In [ ]:
# 月次MPの推移確認
print(f'{TARGET_COLUMN}（月末値）の統計:')
stats = df_monthly[TARGET_COLUMN]
print(f'  平均: {stats.mean():,.0f}  |  標準偏差: {stats.std():,.0f}')
print(f'  最小: {stats.min():,.0f}  |  最大: {stats.max():,.0f}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_monthly[DATE_COLUMN], df_monthly[TARGET_COLUMN] / TARGET_SCALE,
        'b-o', markersize=3, linewidth=1.5)
ax.set_title(f'{TARGET_DISPLAY_NAME} 月末値の推移', fontsize=13)
ax.set_ylabel(f'{TARGET_DISPLAY_NAME}（{TARGET_UNIT}）')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. 特徴量エンジニアリング

**Xか月先予測のリーク防止**: ラグ特徴量は `FORECAST_HORIZON` か月以上過去の値のみ使用します。

| 特徴量グループ | 内容 |
|---|---|
| ラグ特徴量 | TARGET の lag[N] 〜 lag[N+11]（N=FORECAST_HORIZON）|
| 他列のラグ | 他の数値列の lag[N] |
| ローリング統計 | 3/6/12か月移動平均・標準偏差（リークなし） |
| 時間特徴量 | 年・月・四半期・sin/cos変換・フラグ変数 |

In [ ]:
# 特徴量として使う列
exclude_list = [DATE_COLUMN] + EXCLUDE_COLUMNS
raw_feature_cols = [c for c in df_monthly.columns if c not in exclude_list]

print(f'元データの列（特徴量候補）: {len(raw_feature_cols)}列')
print(raw_feature_cols)

In [ ]:
def build_features(df_in, date_col, target_col, forecast_horizon, raw_feat_cols):
    df = df_in.copy().sort_values(date_col).reset_index(drop=True)
    feat_names = []

    # ── ターゲットのラグ（lag N 〜 N+11）──
    for lag in range(forecast_horizon, forecast_horizon + 12):
        col = f'{target_col}_lag{lag}'
        df[col] = df[target_col].shift(lag)
        feat_names.append(col)

    # ── 他列のラグ（lag N のみ）──
    for c in raw_feat_cols:
        if c == target_col:
            continue
        col = f'{c}_lag{forecast_horizon}'
        df[col] = df[c].shift(forecast_horizon)
        feat_names.append(col)

    # ── ローリング統計（リークなし: shift(forecast_horizon) してから rolling）──
    shifted = df[target_col].shift(forecast_horizon)
    for window in [3, 6, 12]:
        col_mean = f'{target_col}_roll{window}m_mean'
        col_std  = f'{target_col}_roll{window}m_std'
        df[col_mean] = shifted.rolling(window, min_periods=1).mean()
        df[col_std]  = shifted.rolling(window, min_periods=1).std()
        feat_names += [col_mean, col_std]

    # ── 時間特徴量 ──
    min_date = df[date_col].min()
    df['year']              = df[date_col].dt.year
    df['month']             = df[date_col].dt.month
    df['quarter']           = df[date_col].dt.quarter
    df['months_since_start'] = (
        (df[date_col].dt.year - min_date.year) * 12 +
        (df[date_col].dt.month - min_date.month)
    )
    df['month_sin']         = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']         = np.cos(2 * np.pi * df['month'] / 12)
    df['quarter_sin']       = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos']       = np.cos(2 * np.pi * df['quarter'] / 4)
    df['is_q1']             = (df['quarter'] == 1).astype(int)
    df['is_q2']             = (df['quarter'] == 2).astype(int)
    df['is_q3']             = (df['quarter'] == 3).astype(int)
    df['is_q4']             = (df['quarter'] == 4).astype(int)
    df['is_year_start']     = (df['month'] == 1).astype(int)
    df['is_year_end']       = (df['month'] == 12).astype(int)
    df['is_quarter_start']  = df['month'].isin([1, 4, 7, 10]).astype(int)
    df['is_quarter_end']    = df['month'].isin([3, 6, 9, 12]).astype(int)

    time_feat_names = [
        'year', 'month', 'quarter', 'months_since_start',
        'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos',
        'is_q1', 'is_q2', 'is_q3', 'is_q4',
        'is_year_start', 'is_year_end', 'is_quarter_start', 'is_quarter_end'
    ]
    feat_names += time_feat_names

    # ── 予測ターゲット（forecast_horizon か月後の TARGET 値）──
    df['forecast_target'] = df[target_col].shift(-forecast_horizon)

    return df, feat_names


df_feat, FEATURE_COLS = build_features(
    df_monthly, DATE_COLUMN, TARGET_COLUMN, FORECAST_HORIZON, raw_feature_cols
)

print(f'特徴量数: {len(FEATURE_COLS)}列')
print(f'  ターゲットラグ  : 12列（lag{FORECAST_HORIZON} ～ lag{FORECAST_HORIZON+11}）')
print(f'  他列のラグ      : {len(raw_feature_cols)-1}列（lag{FORECAST_HORIZON}）')
print(f'  ローリング統計  : 6列（3/6/12か月 × 平均/標準偏差）')
print(f'  時間特徴量      : 16列')

## 6. 学習データ・テストデータの分割

In [ ]:
# forecast_target が既知の行 → モデル学習用
df_labeled = (
    df_feat[df_feat['forecast_target'].notna()]
    .dropna(subset=FEATURE_COLS)
    .copy()
)

# forecast_target が NaN の行 → 将来予測用（特徴量は揃っている行のみ）
df_future = (
    df_feat[df_feat['forecast_target'].isna()]
    .dropna(subset=FEATURE_COLS)
    .copy()
)

print(f'学習可能データ: {len(df_labeled)}か月')
print(f'将来予測対象  : {len(df_future)}か月')

if len(df_labeled) <= TEST_MONTHS:
    raise ValueError(
        f'学習データ不足: TEST_MONTHS={TEST_MONTHS} に対して学習可能データが {len(df_labeled)} か月しかありません。'
        f' TEST_MONTHS を減らすか、データを増やしてください。'
    )

# 時系列順に末尾 TEST_MONTHS か月をテストに使用
train_df = df_labeled.iloc[:-TEST_MONTHS].copy()
test_df  = df_labeled.iloc[-TEST_MONTHS:].copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df['forecast_target']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['forecast_target']

# 予測対象月（基準日 + FORECAST_HORIZON か月）
test_target_dates = test_df[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON)

print(f'\n学習期間  : {train_df[DATE_COLUMN].min().strftime("%Y-%m")} ～ {train_df[DATE_COLUMN].max().strftime("%Y-%m")} ({len(train_df)}か月)')
print(f'テスト期間: {test_df[DATE_COLUMN].min().strftime("%Y-%m")} ～ {test_df[DATE_COLUMN].max().strftime("%Y-%m")} ({len(test_df)}か月)')
print(f'  → テストの予測対象月: {test_target_dates.min().strftime("%Y-%m")} ～ {test_target_dates.max().strftime("%Y-%m")}')

## 7. モデル学習・評価（モデル1〜7）

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape, 'R2': r2}

In [ ]:
models = {
    '1. Linear Regression' : LinearRegression(),
    '2. Ridge Regression'  : Ridge(alpha=1.0),
    '3. Lasso Regression'  : Lasso(alpha=1.0),
    '4. Random Forest'     : RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    '5. Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    '6. XGBoost'           : xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0),
    '7. LightGBM'          : lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1),
}

results     = []
predictions = {}

print(f'モデル学習開始（{FORECAST_HORIZON}か月先予測）\n')
print('=' * 70)

for name, model in models.items():
    print(f'\n{name} を学習中...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    result = evaluate_model(y_test, y_pred, name)
    results.append(result)
    print(f'  MAPE: {result["MAPE(%)"]:.2f}%  R2: {result["R2"]:.4f}')

print('\n' + '=' * 70)
print('モデル 1〜7 学習完了')

## 8. モデル8: Prophet（時系列専用）

In [ ]:
print('8. Prophet を学習中...')

# Prophet の ds: 予測対象月（= 基準日 + FORECAST_HORIZON か月）
prophet_train = pd.DataFrame({
    'ds': train_df[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON),
    'y' : y_train.values
})
prophet_test_ds = pd.DataFrame({
    'ds': test_df[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON)
})

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.05
)
prophet_model.add_seasonality(name='quarterly', period=91.25, fourier_order=4)
prophet_model.fit(prophet_train)

prophet_forecast = prophet_model.predict(prophet_test_ds)
y_pred_prophet = prophet_forecast['yhat'].values
predictions['8. Prophet'] = y_pred_prophet

result_prophet = evaluate_model(y_test, y_pred_prophet, '8. Prophet')
results.append(result_prophet)
print(f'  MAPE: {result_prophet["MAPE(%)"]:.2f}%  R2: {result_prophet["R2"]:.4f}')

## 9. モデル比較結果

In [ ]:
results_df = pd.DataFrame(results).sort_values('MAPE(%)')

results_display = results_df.copy()
results_display['MAE']     = results_display['MAE'].apply(lambda x: f'{x:,.0f}')
results_display['RMSE']    = results_display['RMSE'].apply(lambda x: f'{x:,.0f}')
results_display['MAPE(%)'] = results_display['MAPE(%)'].apply(lambda x: f'{x:.2f}')
results_display['R2']      = results_display['R2'].apply(lambda x: f'{x:.4f}')

print('=' * 80)
print(f'モデル比較結果（{FORECAST_HORIZON}か月先予測、MAPE昇順）')
print('=' * 80)
print('※ MAPE（平均絶対パーセント誤差）が小さいほど精度が高い\n')
print(results_display.to_string(index=False))

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_mape       = results_df.iloc[0]['MAPE(%)']

print('\n' + '*' * 80)
print(f'最良モデル: {best_model_name}')
print(f'MAPE: {best_mape:.2f}%')
print('*' * 80)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAPE 比較
ax1 = axes[0]
bar_colors = ['gold' if m == best_model_name else 'steelblue' for m in results_df['Model']]
bars = ax1.barh(results_df['Model'], results_df['MAPE(%)'], color=bar_colors)
ax1.set_xlabel('MAPE (%)')
ax1.set_title(f'MAPE比較（{FORECAST_HORIZON}か月先予測）', fontsize=12)
ax1.invert_yaxis()
for bar, val in zip(bars, results_df['MAPE(%)']):
    ax1.text(val + 0.05, bar.get_y() + bar.get_height() / 2, f'{val:.2f}%', va='center', fontsize=9)

# R2 比較
ax2 = axes[1]
bar_colors = ['gold' if m == best_model_name else 'steelblue' for m in results_df['Model']]
bars = ax2.barh(results_df['Model'], results_df['R2'], color=bar_colors)
ax2.set_xlabel('R²')
ax2.set_title('R²比較（高いほど良い）', fontsize=12)
ax2.invert_yaxis()
for bar, val in zip(bars, results_df['R2']):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 10. 予測可視化

In [ ]:
# テスト期間での予測比較（上位3モデル）
top3 = results_df.head(3)['Model'].tolist()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_target_dates, y_test / TARGET_SCALE,
        'k-o', label='実績値', linewidth=2, markersize=5)

line_colors  = ['red', 'blue', 'green']
line_markers = ['s', '^', 'D']
for i, mname in enumerate(top3):
    ax.plot(test_target_dates, predictions[mname] / TARGET_SCALE,
            '--', color=line_colors[i], label=mname,
            linewidth=1.5, alpha=0.8, marker=line_markers[i], markersize=5)

ax.set_title(f'{TARGET_DISPLAY_NAME} {FORECAST_HORIZON}か月先予測 vs 実績（上位3モデル）', fontsize=13)
ax.set_xlabel('予測対象月')
ax.set_ylabel(f'{TARGET_DISPLAY_NAME}（{TARGET_UNIT}）')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 最良モデルの詳細
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
best_pred = predictions[best_model_name]

# 上段: 予測 vs 実績
ax1 = axes[0]
ax1.plot(test_target_dates, y_test / TARGET_SCALE, 'b-o', label='実績値', linewidth=2)
ax1.plot(test_target_dates, best_pred / TARGET_SCALE, 'r--s',
         label=f'{best_model_name} 予測', linewidth=2)
ax1.fill_between(test_target_dates,
                 y_test / TARGET_SCALE, best_pred / TARGET_SCALE,
                 alpha=0.2, color='gray')
ax1.set_title(f'最良モデル（{best_model_name}）: 予測 vs 実績', fontsize=13)
ax1.set_ylabel(f'{TARGET_DISPLAY_NAME}（{TARGET_UNIT}）')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 下段: 予測誤差
ax2 = axes[1]
error_pct  = (best_pred - y_test.values) / y_test.values * 100
bar_colors = ['red' if e > 0 else 'blue' for e in error_pct]
ax2.bar(test_target_dates, error_pct, color=bar_colors, alpha=0.7, width=20)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.axhline(y=error_pct.mean(), color='green', linestyle='--',
            label=f'平均誤差: {error_pct.mean():.2f}%')
ax2.set_title(f'予測誤差（%）— {FORECAST_HORIZON}か月先', fontsize=13)
ax2.set_ylabel('誤差 (%)')
ax2.set_xlabel('予測対象月')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 11. 特徴量重要度（ツリー系モデル）

In [ ]:
lgb_model  = models['7. LightGBM']
importance = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Importance': lgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

top_n = min(20, len(importance))
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance.head(top_n)['Feature'], importance.head(top_n)['Importance'],
        color='steelblue')
ax.set_xlabel('重要度')
ax.set_title(f'特徴量重要度（LightGBM, Top {top_n}）', fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'\n特徴量重要度 Top {top_n}:')
print(importance.head(top_n).to_string(index=False))

## 12. 将来予測（最良モデルで全データ再学習）

In [ ]:
print(f'最良モデル（{best_model_name}）を全学習データで再学習...')

X_full = df_labeled[FEATURE_COLS]
y_full = df_labeled['forecast_target']

if 'Prophet' in best_model_name:
    final_model = Prophet(
        yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
        seasonality_mode='multiplicative', changepoint_prior_scale=0.05
    )
    final_model.add_seasonality(name='quarterly', period=91.25, fourier_order=4)
    final_model.fit(pd.DataFrame({
        'ds': df_labeled[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON),
        'y' : y_full.values
    }))
elif 'LightGBM' in best_model_name:
    final_model = lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)
    final_model.fit(X_full, y_full)
elif 'XGBoost' in best_model_name:
    final_model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
    final_model.fit(X_full, y_full)
elif 'Random Forest' in best_model_name:
    final_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    final_model.fit(X_full, y_full)
elif 'Gradient Boosting' in best_model_name:
    final_model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
    final_model.fit(X_full, y_full)
elif 'Ridge' in best_model_name:
    final_model = Ridge(alpha=1.0)
    final_model.fit(X_full, y_full)
elif 'Lasso' in best_model_name:
    final_model = Lasso(alpha=1.0)
    final_model.fit(X_full, y_full)
else:
    final_model = LinearRegression()
    final_model.fit(X_full, y_full)

print('再学習完了')

In [ ]:
if len(df_future) > 0:
    if 'Prophet' in best_model_name:
        future_ds     = pd.DataFrame({'ds': df_future[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON)})
        future_preds  = final_model.predict(future_ds)['yhat'].values
    else:
        future_preds  = final_model.predict(df_future[FEATURE_COLS])

    future_result = df_future[[DATE_COLUMN]].copy()
    future_result['基準日'] = future_result[DATE_COLUMN].dt.strftime('%Y-%m')
    future_result['予測対象月'] = (
        future_result[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON)
    ).dt.strftime('%Y-%m')
    future_result[f'予測_{TARGET_DISPLAY_NAME}'] = future_preds
    future_result[f'予測_{TARGET_DISPLAY_NAME}_{TARGET_UNIT}'] = (future_preds / TARGET_SCALE).round(2)

    print(f'将来予測結果（{FORECAST_HORIZON}か月先 月末{TARGET_DISPLAY_NAME}）:')
    print(future_result[['基準日', '予測対象月',
                          f'予測_{TARGET_DISPLAY_NAME}_{TARGET_UNIT}']].to_string(index=False))
else:
    print('将来予測対象のデータがありません。')
    print('（最後の行から特徴量（ラグ）が計算できない場合があります）')
    future_preds  = np.array([])
    future_result = pd.DataFrame()

In [ ]:
# 履歴実績 + 将来予測を一枚のグラフで表示
fig, ax = plt.subplots(figsize=(14, 6))

hist_dates  = df_labeled[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON)
hist_values = df_labeled['forecast_target']
ax.plot(hist_dates, hist_values / TARGET_SCALE,
        'b-', label='実績値', linewidth=1.5, alpha=0.7, marker='o', markersize=3)

if len(future_result) > 0:
    future_dates = future_result[DATE_COLUMN] + pd.DateOffset(months=FORECAST_HORIZON)
    ax.plot(future_dates, future_preds / TARGET_SCALE,
            'r--s', label=f'予測値（{best_model_name}）', linewidth=2, markersize=6)
    ax.axvline(x=hist_dates.max(), color='gray', linestyle=':', alpha=0.7, linewidth=2)
    ylims = ax.get_ylim()
    ax.text(hist_dates.max(), ylims[1] * 0.97, ' 予測開始 →',
            verticalalignment='top', fontsize=10, color='gray')

ax.set_title(f'{TARGET_DISPLAY_NAME} 月末値: 実績 + {FORECAST_HORIZON}か月先予測', fontsize=13)
ax.set_xlabel('予測対象月')
ax.set_ylabel(f'{TARGET_DISPLAY_NAME}（{TARGET_UNIT}）')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. 結果のエクスポート

In [ ]:
# モデル比較結果
results_df.to_csv('model_comparison.csv', index=False, encoding='utf-8-sig')
print('model_comparison.csv — モデル比較結果')

# テスト期間の全モデル予測値
test_export = test_df[[DATE_COLUMN]].copy()
test_export['予測対象月'] = test_target_dates.dt.strftime('%Y-%m')
test_export[f'実績_{TARGET_DISPLAY_NAME}'] = y_test.values
for mname, preds in predictions.items():
    test_export[f'予測_{mname}'] = preds
test_export.to_csv('test_predictions.csv', index=False, encoding='utf-8-sig')
print('test_predictions.csv — テスト期間の予測詳細')

# 将来予測
if len(future_result) > 0:
    future_export = future_result[[
        '基準日', '予測対象月',
        f'予測_{TARGET_DISPLAY_NAME}',
        f'予測_{TARGET_DISPLAY_NAME}_{TARGET_UNIT}'
    ]].copy()
    future_export.to_csv('future_forecast.csv', index=False, encoding='utf-8-sig')
    print('future_forecast.csv — 将来予測結果')

# Google Colab からダウンロード
for fname in ['model_comparison.csv', 'test_predictions.csv', 'future_forecast.csv']:
    try:
        files.download(fname)
    except Exception:
        pass

## 14. サマリー

In [ ]:
print('\n' + '=' * 80)
print('分析サマリー')
print('=' * 80)

print('\n【データ概要】')
print(f'  日次入力データ : {len(df_raw):,}行')
print(f'  月次集約データ : {len(df_monthly)}か月',
      f'({df_monthly[DATE_COLUMN].min().strftime("%Y-%m")}',
      f'～ {df_monthly[DATE_COLUMN].max().strftime("%Y-%m")})')
print(f'  集約方法       : 各月末日の値')
print(f'  予測対象       : {TARGET_COLUMN}（{FORECAST_HORIZON}か月先の月末値）')
print(f'  特徴量数       : {len(FEATURE_COLS)}列')

print('\n【モデル比較結果（MAPE昇順）】')
print(results_display.to_string(index=False))

print(f'\n【最良モデル】')
print(f'  {best_model_name}  /  MAPE: {best_mape:.2f}%')

if len(future_result) > 0:
    print(f'\n【将来予測（{FORECAST_HORIZON}か月先 月末{TARGET_DISPLAY_NAME}）】')
    print(future_result[[
        '基準日', '予測対象月',
        f'予測_{TARGET_DISPLAY_NAME}_{TARGET_UNIT}'
    ]].to_string(index=False))

print('\n' + '=' * 80)